# Reproducing Starkweather, Gershman & Uchida (2018) from CRCNS vta-1

Paper: Starkweather CK, Gershman SJ, Uchida N (2018). The Medial Prefrontal Cortex Shapes Dopamine
Reward Prediction Errors under State Uncertainty. *Neuron* 98, 616-629.
[doi:10.1016/j.neuron.2018.03.036](https://doi.org/10.1016/j.neuron.2018.03.036)

Dataset: CRCNS vta-1, [doi:10.6080/K01J97X6](http://dx.doi.org/10.6080/K01J97X6), streamed from the
public S3 bucket `crcnsarchive` (`vta-1/live/data.zip`).

Version: CRCNS does not version its data sets, so the version is `unversioned`, the only one there
is (the paper cites this DOI, its Data Availability). The data are pinned by the MD5 of the file read,
`data.zip` c5d2a9f8aaf699cbdd2bda1bace22e7e, as given in the data set's `checksums.md5`; the cell
below checks the bytes read against it, so a later change of the file is detected. The data
description read is version 0.6 (March 28, 2018).

Claims: `claims/starkweather-2018.yaml`, written from the version of record with its STAR Methods
and Supplemental Information, and checked by a second reader. The belief state model claims use the
paper's code, https://github.com/cstarkweather/Starkweather2018 at commit d667730, ported to Python
in `belief_td.py` (MATLAB cannot be run here).

| Claim | Tested here |
|---|---|
| fig2-kord-suppresses-mpfc, figs1e-kord-abolishes-late-activity, figs6ce-kord-task1-animal | Blocked: mPFC recordings are not in vta-1 |
| fig3g-anticipatory-licking, figs3-task1-licks-more-sooner, figs3-salb-spares-licking | Blocked: licks are not in vta-1 |
| figs4-cell-types-unchanged | Blocked: only identified dopamine neurons are in vta-1 |
| figs5b-task2-salb-below-free-water, figs5c-odorbc-normalized, anova-slopes-task-drug, anova-time-task-drug-interaction | Blocked: free water deliveries are not in vta-1 |
| figs6-salb-without-kord | Blocked: the no-KORD control neurons are not in vta-1 |
| fig4a, fig4b, fig4b-odorc, fig4ab-saline-pre, fig4c, fig4d, fig4cd-salb-pre | Tested |
| fig5f-task2-slope-signs, fig5e/fig5f significant slopes, fig4e/fig5e slope distributions, fig5f/fig5e slope variances | Tested |
| fig6ab-odorc-above-odorb, fig6cd-omission-dips-intact, figs5d-omission-unchanged-by-salb | Tested |
| fig7cd-model-intact, fig7fgh-model-belief-impaired, fig7ijk-model-timing-impaired | Tested, with the model ported to Python |

In [ ]:
import resource
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import vta
import belief_td

T0 = time.time()
md5_file, etag = vta.checksums()
neurons, md5_read, members = vta.load()
print("checksums.md5 data.zip:", md5_file["data.zip"])
print("S3 ETag of data.zip:   ", etag)
print("MD5 of the bytes read: ", md5_read)
assert md5_read == md5_file["data.zip"] == vta.EXPECTED_MD5
units = vta.neuron_table(neurons)
TR = {w: vta.trials(neurons, units, w) for w in ("file", "40min")}
tr = TR["file"]
print(f"loaded in {time.time() - T0:.1f} s")

## Data check

What the files hold, against the data description and the paper. The two readings of "the 40
minutes following recording onset" (Methods, Data analysis) are `file` (every trial in the file) and
`40min` (odor onset within 40 min of the session's first spike); see `vta.py`.

In [ ]:
print("data.zip holds %d members: %d neuron files (.mat) and the others %s" % (
    len(members), sum(m.endswith(".mat") for m in members), [m for m in members if not m.endswith(".mat")]))
print("member names by prefix:", pd.Series([m.split("/")[-1].rsplit("_", 1)[0] for m in members if m.endswith(".mat")]).value_counts().to_dict())
print("variables in every file:", sorted({k for d in neurons.values() for k in d}))
print("\nneurons by task and drug (paper: task 1 saline 42, task 1 SalB 42, task 2 saline 41, task 2 SalB 47)")
print(units.groupby(["task", "drug"]).agg(neurons=("neuron", "size"), sessions=("session", "nunique")))
print("sessions:", units.session.nunique(), "; neurons per session:",
      units.groupby("session").size().value_counts().sort_index().to_dict())
print("sessions holding both drugs or both tasks:",
      int((units.groupby("session")[["drug", "task"]].nunique() > 1).any(axis=1).sum()))
print("trials per session: %d to %d (paper: 150 to 300)" % (units.groupby("session").n_trials.first().min(),
                                                              units.groupby("session").n_trials.first().max()))

# trial types and their ISIs (ms), against the data description
raw_types = np.concatenate([d["trial_type"] for d in neurons.values()])
print("\ntrials with a missing (NaN) trial_type: %d, in %d files, %d sessions" % (
    np.isnan(raw_types).sum(),
    sum(np.isnan(d["trial_type"]).any() for d in neurons.values()),
    units[[np.isnan(neurons[n]["trial_type"]).any() for n in units.neuron]].session.nunique()))
nan_isi = np.concatenate([(d["reward_onsets"] - d["odor_onsets"])[np.isnan(d["trial_type"])]
                          for d in units.drop_duplicates("session").neuron.map(neurons)])
print("  their reward delays, one per session trial (ms):", np.round(np.sort(nan_isi)).tolist())
s1 = tr.drop_duplicates(["session", "trial"])
summ = s1.groupby("type").agg(trials=("trial", "size"), isi_min=("isi_ms", "min"), isi_max=("isi_ms", "max"),
                              reward_missing=("reward", lambda r: int(r.isna().sum())))
summ["described ISI (s)"] = [vta.ISI_S.get(t, {10: 1.2, 11: 2.8}.get(t, "none")) for t in summ.index]
print("\none row per session trial, by trial type:")
print(summ.to_string())
print("\nby task: types present")
print(s1.groupby("task").type.apply(lambda t: sorted(t.unique())).to_string())

In [ ]:
# time spans, units and ranges
span = s1.groupby("session").agg(first_odor_min=("minutes", "min"), last_odor_min=("minutes", "max"))
print("odor onsets relative to the session's first spike (min): first %.2f to %.2f, last %.2f to %.2f" % (
    span.first_odor_min.min(), span.first_odor_min.max(), span.last_odor_min.min(), span.last_odor_min.max()))
dur = s1.groupby("session").odor.agg(lambda o: (o.max() - o.min()) / 60000)
print("span of odor onsets per session: %.1f to %.1f min" % (dur.min(), dur.max()))
for w in ("file", "40min"):
    print(f"trials in reading {w}: {len(TR[w])} neuron-trials, {len(TR[w].drop_duplicates(['session', 'trial']))} session trials")
gap = s1.sort_values(["session", "odor"]).groupby("session").odor.diff().dropna() / 1000
print("odor onset to next odor onset: median %.1f s, mean %.1f s, range %.1f to %.1f s "
      "(ITI drawn with mean 12 s, plus the trial)" % (gap.median(), gap.mean(), gap.min(), gap.max()))
rates = units.neuron.map(lambda n: len(neurons[n]["spiketimes"]) /
                         ((neurons[n]["spiketimes"][-1] - neurons[n]["spiketimes"][0]) / 1000))
print("mean firing rate per neuron: %.2f to %.2f spikes/s (median %.2f)" % (rates.min(), rates.max(), rates.median()))
for c in ["base", "post", "pre400", "pre200", "omit_raw"]:
    v = tr[c].dropna()
    print(f"{c:9s} spikes/s: n={len(v)}, range {v.min():.1f} to {v.max():.1f}, mean {v.mean():.2f}")
unsorted = [n for n, d in neurons.items() if np.any(np.diff(d["spiketimes"]) < 0) or np.any(np.diff(d["odor_onsets"]) <= 0)]
print("files with unsorted spike or odor times:", unsorted)

In [ ]:
# Recordings deposited more than once: identical spike trains, or pairs in a session sharing most spikes
same = units.groupby(units.neuron.map(lambda n: neurons[n]["spiketimes"].tobytes())).neuron.apply(list)
print("groups of files with identical spike trains:", [g for g in same if len(g) > 1])
def frac_near(x, y, tol=0.5):
    """Fraction of spikes in x within tol ms of a spike in y."""
    k = np.searchsorted(y, x)
    near = np.minimum(np.abs(x - y[np.clip(k, 0, len(y) - 1)]), np.abs(x - y[np.clip(k - 1, 0, len(y) - 1)]))
    return (near <= tol).mean()

pairs = []
for sess, g in units.groupby("session"):
    names = g.neuron.tolist()
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = neurons[names[i]]["spiketimes"], neurons[names[j]]["spiketimes"]
            pairs.append((names[i], names[j], frac_near(a, b), frac_near(b, a), frac_near(a, b + 1000)))
pairs = pd.DataFrame(pairs, columns=["a", "b", "a_in_b", "b_in_a", "a_in_b_shifted_1s"])
print(f"pairs recorded in the same session: {len(pairs)}; fraction of spikes of a within 0.5 ms of a spike of b: "
      f"median {pairs.a_in_b.median():.3f}; with b shifted by 1 s (chance): median {pairs.a_in_b_shifted_1s.median():.3f}, "
      f"max {pairs.a_in_b_shifted_1s.max():.3f}")
print(pairs.assign(top=pairs[["a_in_b", "b_in_a"]].max(axis=1)).nlargest(5, "top").drop(columns="top").round(3).to_string(index=False))
# pairs sharing more than half of either neuron's spikes: one neuron sorted twice; the file with fewer spikes is
# left out in the "count each recording once" variant
dup = pairs[(pairs.a_in_b > 0.5) | (pairs.b_in_a > 0.5)]
DROP = [a if len(neurons[a]["spiketimes"]) < len(neurons[b]["spiketimes"]) else b for a, b in zip(dup.a, dup.b)]
print("pairs sharing more than half of one neuron's spikes:", len(dup))
for a, b in zip(dup.a, dup.b):
    print(f"  {a} ({len(neurons[a]['spiketimes'])} spikes) and {b} ({len(neurons[b]['spiketimes'])} spikes)")
print("left out in the count-once variant:", DROP)
TRD = {w: t[~t.neuron.isin(DROP)] for w, t in TR.items()}

### Do spikes and trial events share a clock?

A contrast between conditions: per neuron, the auROC of spike counts in 250 ms windows for rewarded
odors (A to C) against odor D, in windows from 1 s before to 1.5 s after odor onset. A shared clock
gives chance before odor onset and a response after it. A second contrast at the reward: task 2
rewarded against omission trials of odors B and C, 0 to 250 ms after the usual reward time.

In [ ]:
edges = np.arange(-1000, 1500, 250)
rows = []
for n in units.neuron:
    sp = neurons[n]["spiketimes"]
    g = tr[tr.neuron == n]
    cue, nocue = g[g.type.between(1, 11)].odor.values, g[g.type == 16].odor.values
    for e in edges:
        rows.append(dict(neuron=n, start=e, auc=vta.auroc(vta._count(sp, cue, e, e + 250), vta._count(sp, nocue, e, e + 250))))
clk = pd.DataFrame(rows)
tab = clk.groupby("start").auc.agg(["mean", lambda a: (a > 0.5).mean()]).rename(columns={"<lambda_0>": "frac>0.5"})
print("odor A-C against odor D, auROC by window start (ms from odor onset), over", clk.neuron.nunique(), "neurons")
print(tab.round(3).to_string())
rows = []
t2 = tr[tr.task == 2]
for (n, odor), (rew, om, isi) in {(n, o): (r, m, i) for o, r, m, i in [("B", 10, 14, 1200), ("C", 11, 15, 2800)]
                                   for n in t2.neuron.unique()}.items():
    sp = neurons[n]["spiketimes"]
    g = t2[t2.neuron == n]
    a = vta._count(sp, g[g.type == rew].odor.values + isi, 0, 250)
    b = vta._count(sp, g[g.type == om].odor.values + isi, 0, 250)
    b_pre = vta._count(sp, g[g.type == om].odor.values + isi, -2000, -1750)
    a_pre = vta._count(sp, g[g.type == rew].odor.values + isi, -2000, -1750)
    rows.append(dict(neuron=n, odor=odor, auc_reward=vta.auroc(a, b), auc_shifted=vta.auroc(a_pre, b_pre)))
rw = pd.DataFrame(rows).dropna()
print("\ntask 2, rewarded against omission trials, 0-250 ms after the usual reward time:")
print(rw.groupby("odor")[["auc_reward", "auc_shifted"]].mean().round(3).to_string())
print("(auc_shifted: the same contrast 2 s earlier, before odor onset for odor B)")

**Data check.** data.zip holds 173 members: the 172 neuron files and the directory entry `data/`,
so no lick, free water or mPFC file is skipped by the loader. The 172 files are the four groups of the paper's Figure 4 (42, 42, 41 and 47 neurons)
in 77 sessions, recovered from identical odor onset arrays; no session mixes drugs or tasks. They hold
only spikes, odor and reward times and trial types, in ms. The reward delays of every trial type
match the data description (odor A 1.2 to 2.8 s in steps of 0.2 s, odor B 1.2 s, odor C 2.8 s; none on
omission and odor D trials). 21 file trials (9 session trials, in 8 sessions) are rewarded but have no
trial type, at delays off the 0.2 s grid (1471 to 2421 ms); they are left out. By group they are 5
(task 1 saline), 1 (task 1 SalB), 1 (task 2 saline) and 14 (task 2 SalB) neuron trials. Kept as odor A
trials at the nearest grid ISI, they change no Figure 4 ANOVA's conclusion (the largest changes are
task 2 SalB pre-reward 0 to 200 ms, F 7.72 to 7.59, and task 2 SalB post-reward, F 0.70 to 0.68, p 0.69
to 0.71) and no slope count of Figure 5 (the reading `file-untyped-kept` below). The sessions hold 137 to 231 trials,
below the paper's 150 to 300 in some, and their trials span 23.8 to 38.3 min, which fits files already
cut to the 40 minutes the paper analyses; the stricter 40-minute reading drops 55 of 34430 neuron trials.
Spikes and events share a clock: the auROC of rewarded odors against odor D is 0.496 to 0.513 in the
windows before odor onset and 0.645 in the 250 ms after it, and rewarded against omission trials
differ at the usual reward time (auROC 0.87 and 0.86) but not 2 s earlier (0.50 and 0.48). No two
files hold the same spike train, but three pairs of task 1 SalB files share 65 to 99.7% of their spikes
within 0.5 ms (chance, with one train shifted by 1 s, at most 1.5%): each pair is very likely one neuron
deposited twice. The paper's counts include all six files, so the analyses below use every file, and
the task 1 SalB claims also show a variant counting each pair once (39 neurons).

## Figure 4: temporal modulation of post- and pre-reward responses

Per neuron and ISI, the mean over rewarded odor A trials of the post-reward response (50 to 200 ms
after reward) or pre-reward response (0 to 400 ms, or 0 to 200 ms, before reward), each minus the
trial's baseline (1 s before odor onset), in spikes/s; additive two-way ANOVA (ISI, neuron) and the
slope of the neuron-averaged response on ISI (claims file, opening comment and criteria). Both
40-minute readings are run. A by-session breakdown follows (no subject identifiers are in the data).

In [ ]:
def anova_table(measures=("post", "pre400", "pre200")):
    rows = []
    for w in ("file", "40min"):
        for task in (1, 2):
            for drug in ("Saline", "SalB"):
                s = TR[w][(TR[w].task == task) & (TR[w].drug == drug)]
                for m in measures:
                    r = vta.isi_anova(s, m)
                    rows.append(dict(window=w, task=task, drug=drug, measure=m, n=r["n"],
                                     F=r["F"], df=f"({r['df1']}, {r['df2']})", p=r["p"], slope=r["slope"]))
    return pd.DataFrame(rows)

A4 = anova_table()
pd.set_option("display.width", 160)
print(A4.to_string(index=False, formatters={"F": "{:.2f}".format, "p": "{:.2g}".format, "slope": "{:+.2f}".format}))

In [ ]:
# Trials without a trial type (left out above): per group, and the Figure 4 ANOVAs and Figure 5 slope counts
# with each kept as an odor A trial at the grid ISI nearest its reward delay (vta.trials, untyped="nearest")
TRN = vta.trials(neurons, units, "file", untyped="nearest")
u = TRN[TRN.untyped]
print("untyped trials kept, by group (neuron trials):", u.groupby(["task", "drug"]).size().to_dict())
print("their reward delays (ms) and assigned ISI (s):", sorted(set(zip(u.isi_ms.round().astype(int), u.isi))))
rows = []
for task in (1, 2):
    for drug in ("Saline", "SalB"):
        for m in ("post", "pre400", "pre200"):
            a = vta.isi_anova(TR["file"][(TR["file"].task == task) & (TR["file"].drug == drug)], m)
            b = vta.isi_anova(TRN[(TRN.task == task) & (TRN.drug == drug)], m)
            rows.append(dict(task=task, drug=drug, measure=m, F_left_out=a["F"], p_left_out=a["p"], slope_left_out=a["slope"],
                             F_kept=b["F"], p_kept=b["p"], slope_kept=b["slope"]))
print(pd.DataFrame(rows).to_string(index=False, float_format="{:.3g}".format))


In [ ]:
# neuron-averaged responses by ISI (spikes/s), reading "file"
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (task, drug) in zip(axes, [(1, "Saline"), (2, "Saline"), (1, "SalB"), (2, "SalB")]):
    s = tr[(tr.task == task) & (tr.drug == drug)]
    for m, c in [("post", "C0"), ("pre400", "k")]:
        b = vta.isi_anova(s, m)["by_isi"]
        ax.plot(b.index, b.values, "o-", color=c, label=m)
        print(f"task {task} {drug} {m}: " + ", ".join(f"{i:.1f}s {v:.2f}" for i, v in b.items()))
    ax.set_title(f"task {task} {drug}")
    ax.set_xlabel("ISI (s)")
axes[0].set_ylabel("baseline-subtracted spikes/s")
axes[0].legend()
plt.tight_layout()
print("all plotted points are within the axes")

In [ ]:
def session_signs(measure):
    """Per session, slope of the session's neuron-averaged response on ISI."""
    rows = []
    for (task, drug, sess), s in tr.groupby(["task", "drug", "session"]):
        a = s[s.type.between(1, 9)].assign(y=lambda d: d[measure])
        by = a.groupby(["neuron", "isi"]).y.mean().groupby("isi").mean()
        rows.append(dict(task=task, drug=drug, session=sess, slope=stats.linregress(by.index, by.values).slope))
    d = pd.DataFrame(rows)
    return d.groupby(["task", "drug"]).slope.agg(sessions="size", negative=lambda x: int((x < 0).sum()),
                                                 positive=lambda x: int((x > 0).sum()), median="median")

print("post-reward, by session:\n", session_signs("post").round(2).to_string())
print("\npre-reward 0-400 ms, by session:\n", session_signs("pre400").round(2).to_string())

### fig4a-task1-saline-post-decrease

> In task 1 under saline, dopamine post-reward responses on odor A trials decreased as the ISI grew (42 neurons).

In [ ]:
sel = A4[(A4.task == 1) & (A4.drug == "Saline") & (A4.measure == "post")]
print(sel.to_string(index=False))
print("neurons:", units[(units.task == 1) & (units.drug == "Saline")].shape[0])

**Result: reproduced.** On the 42 task 1 saline neurons, the ISI effect on post-reward responses is F(8, 328) = 12.61, p = 8.4e-16 (paper F = 12.6, p = 9.1e-16), and the neuron-averaged response falls by 2.21 spikes/s per s of ISI; with the 40-minute cut, F = 12.44, p = 1.4e-15. In 16 of 20 sessions the slope is negative.

### fig4b-task2-saline-post-increase

> In task 2 under saline, dopamine post-reward responses on odor A trials increased as the ISI grew (41 neurons).

In [ ]:
sel = A4[(A4.task == 2) & (A4.drug == "Saline") & (A4.measure == "post")]
print(sel.to_string(index=False))
print("neurons:", units[(units.task == 2) & (units.drug == "Saline")].shape[0])

**Result: reproduced.** On the 41 task 2 saline neurons, F(8, 320) = 3.70, p = 0.00038 (paper F = 3.7, p = 3.8e-4), with a neuron-averaged slope of +1.30 spikes/s per s, the same under both 40-minute readings. In 15 of 21 sessions the slope is positive.

### fig4b-odorc-below-latest-odora

> In task 2 under saline, the post-reward response to odor C (ISI 2.8 s) was significantly smaller
> than that to odor A rewards at 2.8 s, so ISI length alone does not explain the positive temporal modulation.

Per neuron means on rewarded odor C trials (type 11) and odor A trials rewarded at 2.8 s (type 9),
additive ANOVA (condition, neuron); and the same ANOVA over single trials, the other reading.

In [ ]:
for w in ("file", "40min"):
    s = TR[w][(TR[w].task == 2) & (TR[w].drug == "Saline") & TR[w].type.isin([9, 11])].assign(y=lambda d: d.post)
    cell = s.groupby(["neuron", "type"], as_index=False).y.mean()
    r = vta.two_level_anova(cell, "type")
    rt = vta.two_level_anova(s, "type")
    m = cell.groupby("type").y.mean()
    print(f"[{w}] neuron means: A at 2.8 s {m[9]:.2f}, C {m[11]:.2f} spikes/s (C - A {m[11] - m[9]:.2f}); "
          f"F({r['df1']}, {r['df2']}) = {r['F']:.2f}, p = {r['p']:.2g}; neurons {cell.neuron.nunique()}")
    print(f"[{w}] single trials: F({rt['df1']}, {rt['df2']}) = {rt['F']:.2f}, p = {rt['p']:.2g}; trials A {int((s.type == 9).sum())}, C {int((s.type == 11).sum())}")
    print(f"[{w}] neurons with C below A: {int((cell.pivot(index='neuron', columns='type', values='y').diff(axis=1)[11] < 0).sum())} of {cell.neuron.nunique()}")

**Result: reproduced.** The odor C response (12.30 spikes/s) is 2.76 spikes/s below the odor A response at 2.8 s (15.06), F(1, 40) = 22.66, p = 2.5e-05 on neuron means (paper F(1, 41) = 22.4, p = 2.7e-5), and F(1, 1198) = 27.85, p = 1.6e-07 on single trials; 32 of 41 neurons respond less to odor C.

### fig4ab-saline-pre-decrease

> Under saline, pre-reward firing on odor A trials decreased over time in both tasks.

Both windows (0 to 400 ms, Methods; 0 to 200 ms, Figure 4A legend) and both 40-minute readings.

In [ ]:
print(A4[(A4.drug == "Saline") & A4.measure.isin(["pre400", "pre200"])].to_string(index=False))

**Result: reproduced.** With the Methods' 0 to 400 ms window, the pre-reward ISI effect is F(8, 328) = 9.00, p = 3.6e-11 in task 1 and F(8, 320) = 14.44, p = 5.3e-18 in task 2, with negative slopes (-1.03 and -1.62 spikes/s per s); these match the paper's F > 9.0, p < 3.5e-11 and its task 2 F = 14, p = 5.3e-18. The 0 to 200 ms window and the 40-minute cut give p < 1.2e-07 and negative slopes as well.

### fig4c-task1-salb-post-decrease

> In task 1 under SalB (mPFC inactivation), post-reward responses still decreased as the ISI grew, as under saline (42 neurons).

In [ ]:
sel = A4[(A4.task == 1) & (A4.drug == "SalB") & (A4.measure == "post")]
print(sel.to_string(index=False))
print("neurons:", units[(units.task == 1) & (units.drug == "SalB")].shape[0])

In [ ]:
r = vta.isi_anova(TRD["file"][(TRD["file"].task == 1) & (TRD["file"].drug == "SalB")], "post")
print(f"count-once variant: F({r['df1']}, {r['df2']}) = {r['F']:.2f}, p = {r['p']:.2g}, slope {r['slope']:+.2f}, neurons {r['n']}")

**Result: reproduced.** On the 42 task 1 SalB neurons, F(8, 328) = 11.34, p = 3.3e-14 (paper F = 11, p = 3.3e-14), slope -2.33 spikes/s per s. Counting each of the three doubly deposited units once (39 neurons) gives F(8, 304) = 15.11, p = 1.2e-18, slope -2.76.

### fig4d-task2-salb-flat

> In task 2 under SalB, mPFC inactivation abolished the increase of post-reward responses with ISI;
> the ISI effect was not significant (47 neurons).

A claim of absence, judged on the paper's additive two-way ANOVA at alpha 0.05.

In [ ]:
sel = A4[(A4.task == 2) & (A4.drug == "SalB") & (A4.measure == "post")]
print(sel.to_string(index=False))
print("neurons:", units[(units.task == 2) & (units.drug == "SalB")].shape[0])

**Result: reproduced.** On the 47 task 2 SalB neurons the ISI effect is not significant, F(8, 368) = 0.70, p = 0.69 (paper F = 0.69, p = 0.70); the neuron-averaged slope is -0.27 spikes/s per s, against +1.30 under saline. Both 40-minute readings give the same.

### fig4cd-salb-pre-decrease

> Under SalB, pre-reward responses still decreased over time in both tasks.

In [ ]:
print(A4[(A4.drug == "SalB") & A4.measure.isin(["pre400", "pre200"])].to_string(index=False))
for m in ("pre400", "pre200"):
    r = vta.isi_anova(TRD["file"][(TRD["file"].task == 1) & (TRD["file"].drug == "SalB")], m)
    print(f"count-once variant, task 1 SalB {m}: F({r['df1']}, {r['df2']}) = {r['F']:.2f}, p = {r['p']:.2g}, slope {r['slope']:+.2f}, neurons {r['n']}")

**Result: reproduced.** With the 0 to 400 ms window, task 1 SalB F(8, 328) = 15.50, p = 2.4e-19 and task 2 SalB F(8, 368) = 13.83, p = 1.4e-17, both with negative slopes (-1.37 and -1.50); the paper gives F = 16, p = 2.4e-19 and F = 14, p = 2.7e-17. The 0 to 200 ms window gives p of 1.4e-09 or less, and the count-once variant p = 4.9e-22 in task 1.

## Figures 4E and 5: slopes of individual neurons

Per neuron, a least-squares line of the post-reward response (spikes/s) on ISI (s) over its rewarded
odor A trials; significant when p < 0.05 for the slope (Figure 5 legend). The baseline readings of
the criterion (trial by trial, the neuron's mean baseline, none) and both 40-minute readings are run.
The last two baseline readings give identical slopes (subtracting a constant does not change a slope).

In [ ]:
SL = {(w, b): vta.slopes(TR[w], b) for w in ("file", "40min") for b in ("trial", "mean", "none")}
SL[("file-count-once", "trial")] = vta.slopes(TRD["file"], "trial")
SL[("file-untyped-kept", "trial")] = vta.slopes(TRN, "trial")  # untyped trials kept as odor A (data check)

def counts(S):
    rows = []
    for (task, drug), g in S.groupby(["task", "drug"]):
        sig = g.p < 0.05
        rows.append(dict(task=task, drug=drug, n=len(g), positive=int((g.slope > 0).sum()), negative=int((g.slope < 0).sum()),
                         sig_negative=int((sig & (g.slope < 0)).sum()), sig_positive=int((sig & (g.slope > 0)).sum()),
                         median=g.slope.median(), variance=g.slope.var(ddof=1)))
    return pd.DataFrame(rows).set_index(["task", "drug"])

for key, S in SL.items():
    print(key)
    print(counts(S).round(2).to_string(), "\n")
S = SL[("file", "trial")]
print("trial counts per neuron fit: %d to %d; fits that failed: %d" % (S.n_trials.min(), S.n_trials.max(), int(S.slope.isna().sum())))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, task in zip(axes, (1, 2)):
    for drug, c in (("Saline", "k"), ("SalB", "r")):
        x = S[(S.task == task) & (S.drug == drug)].slope
        ax.hist(x, bins=np.arange(-16, 16, 1), histtype="step", color=c, label=drug)
        print(f"task {task} {drug}: slopes {x.min():.2f} to {x.max():.2f} spikes/s per s")
    ax.set_title(f"task {task}")
    ax.set_xlabel("slope (spikes/s per s)")
axes[0].legend()
plt.tight_layout()
print("all slopes fall within the histogram range -16 to 15")

In [ ]:
# by session (no subject identifiers in the data): sessions whose neurons' median slope is positive
bys = S.groupby(["task", "drug", "session"]).slope.median().reset_index()
print(bys.groupby(["task", "drug"]).slope.agg(sessions="size", positive=lambda x: int((x > 0).sum()),
                                               negative=lambda x: int((x < 0).sum())).to_string())

### fig5f-task2-slope-signs

> In task 2, 29 of 41 neurons (70%) had positive slopes of post-reward response on ISI under saline
> and 12 of 41 negative, while under SalB 24 of 47 (51%) had negative slopes.

Expected 29 ± 2.9, 12 ± 1.2 and 24 ± 2.4; reproduced when under a reading all three are within tolerance.

In [ ]:
for key, Sx in SL.items():
    c = counts(Sx)
    vals = (c.loc[(2, "Saline"), "positive"], c.loc[(2, "Saline"), "negative"], c.loc[(2, "SalB"), "negative"])
    ok = abs(vals[0] - 29) <= 2.9 and abs(vals[1] - 12) <= 1.2 and abs(vals[2] - 24) <= 2.4
    print(key, "saline positive %d, saline negative %d, SalB negative %d of %d; within tolerance: %s" % (
        *vals, c.loc[(2, "SalB"), "n"], ok))

**Result: reproduced.** With trial-by-trial baseline subtraction, 29 of 41 task 2 saline neurons have positive slopes and 12 negative, and 24 of 47 SalB neurons have negative slopes, the paper's three counts exactly. The other baseline readings give 28, 13 and 25, also within tolerance.

### fig5e-task1-significant-slopes-unchanged

> In task 1 the proportion of significant negative against positive slopes did not differ between
> saline (16 negative, 0 positive) and SalB (16 negative, 2 positive).

Counts 16 ± 1.6, 0 ± 1, 16 ± 1.6, 2 ± 1; absence judged on the two-sided Fisher exact test at alpha 0.05.

In [ ]:
for key, Sx in SL.items():
    c = counts(Sx)
    tab = [[c.loc[(1, d), "sig_negative"], c.loc[(1, d), "sig_positive"]] for d in ("Saline", "SalB")]
    odds, p = stats.fisher_exact(tab)
    got = [tab[0][0], tab[0][1], tab[1][0], tab[1][1]]
    within = all(abs(g - e) <= t for g, e, t in zip(got, [16, 0, 16, 2], [1.6, 1, 1.6, 1]))
    share = [t[0] / max(t[0] + t[1], 1) for t in tab]
    print(key, "saline neg/pos %d/%d, SalB neg/pos %d/%d; Fisher two-sided p = %.3g; negative share saline %.2f, SalB %.2f; counts within tolerance: %s"
          % (*got, p, *share, within))
print("printed counts of the paper, Fisher p:", round(stats.fisher_exact([[16, 0, 16, 2][:2], [16, 0, 16, 2][2:]])[1], 4))

**Result: reproduced, resting on trial-by-trial baseline subtraction.** With trial-by-trial baseline, the significant slopes are 16 negative and 0 positive under saline and 16 negative and 2 positive under SalB, the paper's counts, and Fisher's exact test gives p = 0.487 (paper 0.49). With the neuron's mean baseline or none, SalB has 18 significant negative slopes, outside the tolerance of 1.6, while Fisher p = 1 still finds no difference. The count-once variant gives 16/0 against 16/1, p = 1.

### fig5f-task2-significant-slopes-shift

> In task 2 the proportions of significant negative and positive slopes changed under SalB (saline 1
> negative, 9 positive; SalB 9 negative, 7 positive), so neurons were more likely to show significant
> negative modulation under SalB.

Counts within one neuron; reproduced when the two-sided Fisher test has p < 0.05 with a larger
negative share under SalB and the counts are within tolerance.

In [ ]:
for key, Sx in SL.items():
    c = counts(Sx)
    tab = [[c.loc[(2, d), "sig_negative"], c.loc[(2, d), "sig_positive"]] for d in ("Saline", "SalB")]
    odds, p = stats.fisher_exact(tab)
    got = [tab[0][0], tab[0][1], tab[1][0], tab[1][1]]
    within = all(abs(g - e) <= t for g, e, t in zip(got, [1, 9, 9, 7], [1, 1, 1, 1]))
    share = [t[0] / max(t[0] + t[1], 1) for t in tab]
    print(key, "saline neg/pos %d/%d, SalB neg/pos %d/%d; Fisher two-sided p = %.3g; negative share saline %.2f, SalB %.2f; counts within tolerance: %s"
          % (*got, p, *share, within))
print("printed counts of the paper, Fisher p:", round(stats.fisher_exact([[1, 9, 9, 7][:2], [1, 9, 9, 7][2:]])[1], 4))

**Result: reproduced, resting on trial-by-trial baseline subtraction.** Under that reading the significant slopes are 1 negative and 9 positive under saline and 8 negative and 7 positive under SalB (paper 9 and 7; all within one neuron), the negative share rises from 0.10 to 0.53, and Fisher's exact test gives p = 0.0405, the paper's p = 0.041 (the paper's printed counts give 0.0367). With the neuron's mean baseline or none, saline has 3 significant negative slopes and p = 0.239, so the claim does not hold under those readings. The outcome therefore rests on the trial-by-trial reading. The Methods say only that responses "were baseline-subtracted, with baseline taken 0-1 s prior to odor onset", which fits either reading. Trial by trial is adopted because it is the only reading that recovers the paper's printed counts listed here (29, 12 and 24 in task 2; 16, 0, 16 and 2 in task 1) and its Fisher p values (0.487 and 0.0405 against 0.49 and 0.041), though in this claim it gives 8 significant negative SalB slopes where the paper prints 9; the mean and none readings give 3 significant negative task 2 saline slopes where the paper prints 1. Keeping the untyped trials or counting the doubly deposited units once gives the same counts and p = 0.0405.

### fig4e-task2-slope-distribution-shift

> In task 2 the distribution of slopes under SalB shifted toward smaller values than under saline.

In [ ]:
for key, Sx in SL.items():
    a = Sx[(Sx.task == 2) & (Sx.drug == "Saline")].slope
    b = Sx[(Sx.task == 2) & (Sx.drug == "SalB")].slope
    r = stats.ranksums(a, b)
    print(key, "medians saline %.2f, SalB %.2f (difference %.2f); rank-sum z = %.2f, p = %.3g; n %d, %d"
          % (a.median(), b.median(), b.median() - a.median(), r.statistic, r.pvalue, len(a), len(b)))

**Result: reproduced.** The median task 2 slope falls from 1.42 under saline to -0.06 under SalB, rank-sum z = 2.15, p = 0.032 (paper z = 2.1, p = 0.038); every reading gives p between 0.032 and 0.037.

### fig5e-task1-slope-distribution-unchanged

> In task 1 the distribution of slopes did not differ between saline and SalB.

A claim of absence, judged on the two-sided rank-sum test at alpha 0.05.

In [ ]:
for key, Sx in SL.items():
    a = Sx[(Sx.task == 1) & (Sx.drug == "Saline")].slope
    b = Sx[(Sx.task == 1) & (Sx.drug == "SalB")].slope
    r = stats.ranksums(a, b)
    print(key, "medians saline %.2f, SalB %.2f (difference %.2f); rank-sum z = %.2f, p = %.3g; n %d, %d"
          % (a.median(), b.median(), b.median() - a.median(), r.statistic, r.pvalue, len(a), len(b)))

**Result: reproduced.** The task 1 medians are -1.74 (saline) and -1.83 (SalB), rank-sum z = 0.14, p = 0.886 (paper z = 0.14, p = 0.89); every reading gives p of 0.63 or more.

### fig5f-task2-slope-variance-up

> In task 2 the variance of slopes was significantly greater under SalB than under saline (about 20.0
> against 10.2), seen in individual mice as well.

Ratio 0.51 ± 0.051, SalB 20.0 ± 2.0, saline 10.2 ± 1.02 (spikes/s per s)²; two-sided F-test p < 0.05.
The per-mouse inset cannot be made: the files carry no subject identifier. The per-session variances
are shown in its place, for sessions with at least three neurons.

In [ ]:
for key, Sx in SL.items():
    a = Sx[(Sx.task == 2) & (Sx.drug == "Saline")].slope
    b = Sx[(Sx.task == 2) & (Sx.drug == "SalB")].slope
    F = a.var(ddof=1) / b.var(ddof=1)
    d1, d2 = len(a) - 1, len(b) - 1
    p = 2 * min(stats.f.cdf(F, d1, d2), stats.f.sf(F, d1, d2))
    print(key, "variance saline %.2f, SalB %.2f; F(%d, %d) = %.2f, two-sided p = %.3g" % (a.var(), b.var(), d1, d2, F, p))

In [ ]:
v = S[S.task == 2].groupby(["drug", "session"]).slope.agg(["size", "var"])
print(v[v["size"] >= 3].round(2).to_string())

**Result: reproduced.** The task 2 slope variance is 10.23 under saline and 20.29 under SalB (expected 10.2 and 20.0), ratio F(40, 46) = 0.50 (expected 0.51 +- 0.051), two-sided p = 0.029 (paper 0.03). The other baseline readings give 0.52, p = 0.036. The per-mouse inset cannot be made, as no file names its mouse; per session, variances range from 1.2 to 69.4 under SalB and 1.6 to 58.1 under saline, in the sessions with three or more neurons.

### fig5e-task1-slope-variance-unchanged

> In task 1 the variance of slopes was not significantly greater under SalB (about 14.9 against 8.8).

A claim of absence, judged on the two-sided F-test at alpha 0.05; ratio compared with 0.59.

In [ ]:
for key, Sx in SL.items():
    a = Sx[(Sx.task == 1) & (Sx.drug == "Saline")].slope
    b = Sx[(Sx.task == 1) & (Sx.drug == "SalB")].slope
    F = a.var(ddof=1) / b.var(ddof=1)
    d1, d2 = len(a) - 1, len(b) - 1
    p = 2 * min(stats.f.cdf(F, d1, d2), stats.f.sf(F, d1, d2))
    print(key, "variance saline %.2f, SalB %.2f; F(%d, %d) = %.2f, two-sided p = %.3g" % (a.var(), b.var(), d1, d2, F, p))

**Result: reproduced.** The task 1 variances are 8.81 (saline) and 14.89 (SalB), ratio F(41, 41) = 0.59, two-sided p = 0.097 (paper 0.59, p = 0.10), not significant; every reading gives p of 0.053 or more, and the count-once variant p = 0.28.

## Figure 6 and S5D: odors B and C, and omissions

### fig6ab-odorc-above-odorb

> In task 2, post-reward responses after odor C (2.8 s) were slightly larger than after odor B (1.2
> s), under 1 Hz, significant under saline and not under SalB, and the difference was not larger
> under inactivation.

Per neuron means on rewarded odor B (type 10) and C (type 11) trials; additive ANOVA (odor, neuron)
for each drug; per-neuron C minus B compared between drugs by a two-sided rank-sum test. Expected
differences 0.95 ± 0.095 (saline) and 1.0 ± 0.1 (SalB) spikes/s.

In [ ]:
for w in ("file", "40min"):
    diff = {}
    for drug in ("Saline", "SalB"):
        s = TR[w][(TR[w].task == 2) & (TR[w].drug == drug) & TR[w].type.isin([10, 11])].assign(y=lambda d: d.post)
        cell = s.groupby(["neuron", "type"], as_index=False).y.mean()
        r = vta.two_level_anova(cell, "type")
        p = cell.pivot(index="neuron", columns="type", values="y")
        diff[drug] = p[11] - p[10]
        print(f"[{w}] {drug}: B {p[10].mean():.2f}, C {p[11].mean():.2f} spikes/s, C - B {diff[drug].mean():.3f}; "
              f"F({r['df1']}, {r['df2']}) = {r['F']:.2f}, p = {r['p']:.3g}; neurons {len(p)}")
    r = stats.ranksums(diff["Saline"], diff["SalB"])
    print(f"[{w}] C - B, saline against SalB: rank-sum z = {r.statistic:.2f}, p = {r.pvalue:.3g}")

**Result: reproduced.** Odor C minus odor B is 0.913 spikes/s under saline (expected 0.95 +- 0.095), F(1, 40) = 5.31, p = 0.027, and 0.997 spikes/s under SalB (expected 1.0 +- 0.1), F(1, 46) = 2.59, p = 0.114, where the paper prints F(1, 40) = 2.6, p = 0.11. The per-neuron differences do not differ between drugs, rank-sum z = 0.24, p = 0.81 (paper p = 0.76).

### fig6cd-omission-dips-intact

> On task 2 omission trials of odors B and C, dopamine neurons paused at the usual reward time, below
> baseline, under both saline and SalB.

Per neuron, rate 0 to 1 s after the usual reward time (1.2 s for B, 2.8 s for C) and in the 1 s
before odor onset on the same trials; additive ANOVA (window, neuron) for each drug and odor.

In [ ]:
for w in ("file", "40min"):
    for drug in ("Saline", "SalB"):
        for ty, odor in ((14, "B"), (15, "C")):
            s = TR[w][(TR[w].task == 2) & (TR[w].drug == drug) & (TR[w].type == ty)]
            g = s.groupby("neuron")[["omit_raw", "base"]].mean()
            cell = pd.concat([pd.DataFrame(dict(neuron=g.index, win="omission", y=g.omit_raw)),
                              pd.DataFrame(dict(neuron=g.index, win="baseline", y=g.base))])
            r = vta.two_level_anova(cell, "win")
            n_tr = s.groupby("neuron").size()
            print(f"[{w}] {drug} odor {odor}: omission {g.omit_raw.mean():.2f}, baseline {g.base.mean():.2f} spikes/s; "
                  f"F({r['df1']}, {r['df2']}) = {r['F']:.1f}, p = {r['p']:.2g}; neurons {len(g)}, "
                  f"trials per neuron {n_tr.min()} to {n_tr.max()}; neurons below baseline {int((g.omit_raw < g.base).sum())}")

**Result: reproduced.** In the 1 s after the usual reward time on omission trials, the rate falls below the pre-odor baseline in all four cases: saline odor B 3.58 against 6.20 spikes/s, F(1, 40) = 36.8, p = 3.9e-07; saline odor C F(1, 40) = 13.6, p = 0.00068; SalB odor B F(1, 46) = 12.7, p = 0.00087; SalB odor C F(1, 46) = 11.8, p = 0.0013. These are close to the paper's four F and p values (37, 14, 13 and 12); the paper prints F(1, 40) for SalB too, where its 47 neurons give 46 degrees of freedom. Each neuron has only 2 to 4 omission trials of each odor.

### figs5d-omission-unchanged-by-salb

> In task 2, baseline-subtracted omission responses in 200 ms epochs did not differ between saline
> and SalB for any trial type.

Per neuron, baseline-subtracted rate in 30 epochs of 200 ms from odor onset to 6 s, on omission
trials of A (13), B (14) and C (15). Readings: rank-sum test or one-way ANOVA between drugs, Bonferroni
over all 90 epochs and trial types, or over the 30 epochs of each trial type.

In [ ]:
for w in ("file", "40min"):
    E = vta.epoch_rates(neurons, TR[w][TR[w].task == 2], [13, 14, 15])
    res = []
    for (ty, e), g in E.groupby(["type", "epoch"]):
        a, b = g[g.drug == "Saline"].y, g[g.drug == "SalB"].y
        res.append(dict(type=ty, epoch=e, diff=b.mean() - a.mean(), p_ranksum=stats.ranksums(a, b).pvalue,
                        p_anova=stats.f_oneway(a, b).pvalue, n_saline=len(a), n_salb=len(b)))
    R = pd.DataFrame(res)
    print(f"[{w}] epochs tested: {len(R)}; neurons saline {R.n_saline.min()} to {R.n_saline.max()}, SalB {R.n_salb.min()} to {R.n_salb.max()}")
    for test in ("p_ranksum", "p_anova"):
        print(f"[{w}] {test}: smallest uncorrected p = {R[test].min():.3g}; significant at Bonferroni over 90: "
              f"{int((R[test] * 90 < 0.05).sum())}, over 30 per type: {int((R[test] * 30 < 0.05).sum())}; uncorrected p < 0.05: {int((R[test] < 0.05).sum())}")
    print(f"[{w}] largest SalB - saline difference: {R['diff'].abs().max():.2f} spikes/s")
    print(R.nsmallest(3, "p_ranksum").round(4).to_string(index=False))

**Result: reproduced.** Of the 90 epochs and trial types, none differs between saline and SalB after Bonferroni correction, whether the correction spans all 90 or the 30 epochs of a trial type and whether the test is the rank-sum test or a one-way ANOVA. The smallest uncorrected p is 0.023, and 3 of the 90 rank-sum tests have uncorrected p < 0.05.

## Figure 7: the belief state TD model

A Python port of the paper's MATLAB code at commit d667730 (`belief_td.py`). Weights are trained once
for each odor (500 sessions of 50 trials, 10 for B and C; Weber fraction 0.05, jitter 2 sub-states,
gamma 0.93, learning rate 0.1), then 50 simulations are run for each condition with awgn noise at
SNR 12. The omission response is the mean of the averaged omission trace over the 5 sub-states (1 s)
from the usual reward time, the span of the data's omission window (Methods, Data analysis); its
minimum near that time is printed too. Each condition is also run under five more seeds, to show how
much a 50-simulation run varies, and the code's 10 simulations as a variant.

In [ ]:
t = time.time()
rng_train = np.random.default_rng(20180502)
W = {o: belief_td.train(o, rng_train) for o in ("A100", "A90", "B", "C")}
print(f"training: {time.time() - t:.0f} s")

def impaired_mask(rng, n=50, k=30):
    return rng.permutation(np.r_[np.ones(k, int), np.zeros(n - k, int)])

CONDS = {
    "intact": dict(),
    "belief": dict(impaired="mask"),
    "timing_a": dict(weber=lambda g: 0.5 + g.random()),
    "timing_b": dict(weber=lambda g: 0.56),
}

def run_cond(name, seed, n_sim=50):
    rng = np.random.default_rng(seed)
    kw = dict(CONDS[name])
    if kw.get("impaired") == "mask":
        kw["impaired"] = impaired_mask(rng, n_sim, int(round(0.6 * n_sim)))
    r = belief_td.summarize(belief_td.run(W, rng, n_sim=n_sim, **kw))
    oB, oC = belief_td.omission_window(r["omission_B"], 6), belief_td.omission_window(r["omission_C"], 14)
    return dict(cond=name, seed=seed, n_sim=n_sim,
                post_t1=r["post_slope_avg_t1"], post_t2=r["post_slope_avg_t2"],
                pre_t1=r["pre_slope_avg_t1"], pre_t2=r["pre_slope_avg_t2"],
                frac_pos_t2=(r["sim_slopes_t2"] > 0).mean(), frac_neg_t2=(r["sim_slopes_t2"] < 0).mean(),
                omitB_mean=oB[0], omitB_min=oB[1], omitC_mean=oC[0], omitC_min=oC[1]), r

t = time.time()
MAIN, MR = {}, {}
for i, name in enumerate(CONDS):
    MAIN[name], MR[name] = run_cond(name, 100 + i)
SEEDS = pd.DataFrame([run_cond(name, 1000 * (i + 1) + s)[0] for i, name in enumerate(CONDS) for s in range(5)])
CODE10 = pd.DataFrame([run_cond(name, 7000 + i, n_sim=10)[0] for i, name in enumerate(["intact", "belief"])])
print(f"simulations: {time.time() - t:.0f} s")
fmt = {k: "{:+.3f}".format for k in ["post_t1", "post_t2", "pre_t1", "pre_t2", "omitB_mean", "omitB_min", "omitC_mean", "omitC_min"]}
print("\nmain runs (slopes in RPE units per s):")
print(pd.DataFrame(MAIN.values()).to_string(index=False, formatters=fmt))
print("\nfive more seeds per condition, range over seeds:")
print(SEEDS.groupby("cond")[list(fmt) + ["frac_pos_t2", "frac_neg_t2"]].agg(["min", "max"]).round(3).T.to_string())
print("\nthe code's 10 simulations (variant):")
print(CODE10.to_string(index=False, formatters=fmt))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for name, c in (("intact", "k"), ("belief", "C1"), ("timing_a", "C2")):
    r = MR[name]
    axes[0].plot(belief_td.ISI_GRID, r["post_avg_t1"], "o-", color=c, label=name)
    axes[1].plot(belief_td.ISI_GRID, r["post_avg_t2"], "o-", color=c, label=name)
    axes[2].plot(np.arange(30) - 5, r["omission_C"], color=c, label=name)
    print(name, "post t1 range %.3f to %.3f, post t2 %.3f to %.3f, omission C %.3f to %.3f" % (
        r["post_avg_t1"].min(), r["post_avg_t1"].max(), r["post_avg_t2"].min(), r["post_avg_t2"].max(),
        r["omission_C"].min(), r["omission_C"].max()))
axes[0].set_title("task 1 post-reward RPE"); axes[1].set_title("task 2 post-reward RPE")
axes[2].set_title("odor C omission RPE (sub-states from cue)"); axes[2].axvline(14, color="0.7")
for a in axes[:2]:
    a.set_xlabel("ISI (s)")
axes[0].legend()
plt.tight_layout()
print("all plotted points are within the axes")

### fig7cd-model-intact

> The belief state TD model reproduces the data with an intact belief state; post-reward RPEs
> decrease with ISI in task 1 and increase in task 2, most task 2 simulations have positive slopes,
> pre-reward RPEs decrease in both tasks, and odors B and C give omission responses at the expected reward time.

In [ ]:
m = MAIN["intact"]
checks = {"task 1 averaged post slope < 0": m["post_t1"] < 0, "task 2 averaged post slope > 0": m["post_t2"] > 0,
          "more than half of task 2 slopes > 0": m["frac_pos_t2"] > 0.5, "pre slopes < 0, both tasks": m["pre_t1"] < 0 and m["pre_t2"] < 0,
          "omission B and C negative": m["omitB_mean"] < 0 and m["omitC_mean"] < 0}
for k, v in checks.items():
    print(f"{k}: {v}")
print({k: round(float(v), 3) for k, v in m.items() if k not in ("cond", "seed", "n_sim")})
s = SEEDS[SEEDS.cond == "intact"]
print("seeds where every part holds:", int(((s.post_t1 < 0) & (s.post_t2 > 0) & (s.frac_pos_t2 > 0.5) & (s.pre_t1 < 0) & (s.pre_t2 < 0)
                                             & (s.omitB_mean < 0) & (s.omitC_mean < 0)).sum()), "of", len(s))

**Result: reproduced, with the model ported to Python.** With an intact belief state, the averaged post-reward slope is -0.114 in task 1 and +0.149 in task 2, 50 of 50 task 2 simulations have positive slopes, the pre-reward slopes are -0.030 and -0.094, and the omission responses of odors B and C are negative in the 1 s from the usual reward time (-0.181 and -0.131). All parts hold in 5 of 5 further seeds and in the code's 10-simulation variant.

### fig7fgh-model-belief-impaired

> Impairing hidden state inference in 60% of simulations recapitulates the inactivation data; task 1
> post-reward RPEs still decrease, task 2 averaged RPEs flatten, more task 2 simulations have negative
> slopes than in the intact model, pre-reward RPEs still decrease, and omission responses remain.

In [ ]:
m, i0 = MAIN["belief"], MAIN["intact"]
checks = {"task 1 averaged post slope < 0": m["post_t1"] < 0,
          "|task 2 averaged post slope| < intact": abs(m["post_t2"]) < abs(i0["post_t2"]),
          "fraction negative task 2 slopes > intact": m["frac_neg_t2"] > i0["frac_neg_t2"],
          "task 2 pre slope < 0": m["pre_t2"] < 0, "omission B and C negative": m["omitB_mean"] < 0 and m["omitC_mean"] < 0}
for k, v in checks.items():
    print(f"{k}: {v}")
print({k: round(float(v), 3) for k, v in m.items() if k not in ("cond", "seed", "n_sim")})
s, s0 = SEEDS[SEEDS.cond == "belief"].reset_index(drop=True), SEEDS[SEEDS.cond == "intact"].reset_index(drop=True)
ok = ((s.post_t1 < 0) & (s.post_t2.abs() < s0.post_t2.abs()) & (s.frac_neg_t2 > s0.frac_neg_t2) & (s.pre_t2 < 0)
      & (s.omitB_mean < 0) & (s.omitC_mean < 0))
print("seeds (paired with the intact seed of the same rank) where every part holds:", int(ok.sum()), "of", len(s))

**Result: reproduced, with the model ported to Python.** With 30 of 50 simulations impaired, the task 1 slope stays negative (-0.154), the task 2 averaged slope shrinks from +0.149 to +0.037, the fraction of negative task 2 slopes rises from 0.00 to 0.32, the task 2 pre-reward slope stays negative (-0.062), and the omission responses stay negative (-0.166 and -0.115). All parts hold in 5 of 5 further seeds.

### fig7ijk-model-timing-impaired

> Impairing timing by blurring the transition matrix flattens task 1 post-reward RPEs and pre-reward
> RPEs in both tasks, leaves most task 2 simulations with positive slopes, and abolishes omission
> responses, unlike the data.

Readings (a) Weber fraction 0.5 + rand per simulation, (b) 0.56 for all. The dip is compared with the
intact run's minimum near the expected time.

In [ ]:
i0 = MAIN["intact"]
for name in ("timing_a", "timing_b"):
    m = MAIN[name]
    checks = {"|task 1 post slope| < intact": abs(m["post_t1"]) < abs(i0["post_t1"]),
              "|pre slopes| < intact, both tasks": abs(m["pre_t1"]) < abs(i0["pre_t1"]) and abs(m["pre_t2"]) < abs(i0["pre_t2"]),
              "more than half of task 2 slopes > 0": m["frac_pos_t2"] > 0.5,
              "omission minimum not below intact's (B and C)": m["omitB_min"] > i0["omitB_min"] and m["omitC_min"] > i0["omitC_min"]}
    print(name, {k: bool(v) for k, v in checks.items()})
    print("  ", {k: round(float(v), 3) for k, v in m.items() if k not in ("cond", "seed", "n_sim")})
s0 = SEEDS[SEEDS.cond == "intact"].reset_index(drop=True)
for name in ("timing_a", "timing_b"):
    s = SEEDS[SEEDS.cond == name].reset_index(drop=True)
    ok = ((s.post_t1.abs() < s0.post_t1.abs()) & (s.pre_t1.abs() < s0.pre_t1.abs()) & (s.pre_t2.abs() < s0.pre_t2.abs())
          & (s.frac_pos_t2 > 0.5) & (s.omitB_min > s0.omitB_min) & (s.omitC_min > s0.omitC_min))
    print(name, "seeds where every part holds:", int(ok.sum()), "of", len(s))
# How deep the remaining omission dip is: per simulation, the mean omission RPE over the 1 s from the usual reward time
# (each simulation has one omission trial per odor), its mean and standard error over the 50 simulations, a one-sample
# t-test against 0, and its size relative to the intact run's mean dip
dip = {}
for name in ("intact", "timing_a", "timing_b"):
    for odor, isi in (("B", 6), ("C", 14)):
        x = MR[name][f"omission_{odor}_sims"][:, isi + 5:isi + 10].mean(1)
        t = stats.ttest_1samp(x, 0)
        dip[(name, odor)] = x.mean()
        print(f"{name:9s} odor {odor}: mean {x.mean():+.3f}, SE {x.std(ddof=1) / np.sqrt(len(x)):.3f}, "
              f"t({len(x) - 1}) = {t.statistic:.2f}, p = {t.pvalue:.2g}, fraction of intact dip {x.mean() / dip.get(('intact', odor), x.mean()):.2f}")


**Result: reproduced, with the model ported to Python.** With a Weber fraction of 0.5 to 1.5 (reading a), the task 1 post-reward slope is +0.011 against -0.114 intact, the pre-reward slopes are +0.005 and +0.040 against -0.030 and -0.094, 50 of 50 task 2 simulations have positive slopes, and the omission minima (-0.110 and -0.090) are less deep than the intact ones (-0.313 and -0.186). Reading (b), 0.56 for all, also meets every part, with a task 1 slope of -0.015 and 49 of 50 task 2 slopes positive, and all parts hold in 5 of 5 further seeds under each reading. The criterion asks only that the dip is not below the intact run's minimum, and that holds. The dips are reduced, not removed: over the 50 simulations the mean omission RPE in the 1 s from the usual reward time is -0.073 (SE 0.014, t(49) = -5.05, p = 6.6e-06) for odor B and -0.032 (SE 0.016, p = 0.045) for odor C under reading (a), and -0.077 (p = 0.00015) and -0.043 (p = 0.019) under reading (b), 25 to 43% of the intact dips (-0.181 and -0.131). So the paper's word "abolished" holds only in the weaker sense the criterion tests; whether the criterion reads the paper rightly is asked of a reader (questions/starkweather-2018.yaml), and the outcome is given on the criterion as written.

## Blocked claims

Every file holds the same four variables (printed in the data check): no licks, no free water
deliveries, no injection times, and only light-identified dopamine neurons of the four KORD groups.

In [ ]:
print("variables:", sorted({k for d in neurons.values() for k in d}))
print("trial types present:", sorted(tr.type.unique()))
print("files by group:", units.groupby(["task", "drug"]).size().to_dict(), "total", len(units))
print("reward onsets on trials that are not odor A to C rewarded trials:",
      int(tr[~tr.type.between(1, 11)].reward.notna().sum()))

| Claim | Blocked by | What is missing |
|---|---|---|
| fig2-kord-suppresses-mpfc | data_missing | mPFC units of the two mPFC-implanted mice |
| figs1e-kord-abolishes-late-activity | data_missing | the same mPFC units, with their odor 3 trials |
| figs6ce-kord-task1-animal | data_missing | mPFC units of the task 1 mouse |
| fig3g-anticipatory-licking | data_missing | licks |
| figs3-task1-licks-more-sooner | data_missing | licks |
| figs3-salb-spares-licking | data_missing | licks |
| figs4-cell-types-unchanged | data_missing | the 761 VTA neurons near identified dopamine neurons |
| figs5b-task2-salb-below-free-water | data_missing | free water deliveries |
| figs5c-odorbc-normalized | data_missing | free water deliveries |
| anova-slopes-task-drug | data_missing | free water deliveries (the slopes are normalized by them) |
| anova-time-task-drug-interaction | data_missing | free water responses, a term of the ANOVA |
| figs6-salb-without-kord | data_missing | the 26 neurons of the two no-KORD mice |

### What of the two blocked ANOVAs can be run

Both ANOVAs need each neuron's free water response, which the files do not hold, so neither can be
tested as the criterion states, and both stay blocked. The parts that do not need it are run here as
a deviation, to show what the data give; they decide no outcome. For anova-time-task-drug-interaction,
the four tr terms without the free water covariate, with task 1, task 2 and drug as 0 or 1 indicators
(Statistics) and tr categorical over the 9 ISIs, on the criterion's two readings, (a) one mean per
neuron and ISI and (b) single rewarded odor A trials; the drug terms take 9 columns, one per ISI, as the
paper's 9 degrees of freedom imply. For anova-slopes-task-drug, the terms task, task 1 x drug and task 2
x drug on the slopes of Figure 5 (trial-by-trial baseline) without normalizing by the free water
response, all 172 neurons.


In [ ]:
grid = np.round(np.arange(1.2, 2.81, 0.2), 1)
def time_task_drug(d):
    t1 = (d.task == 1).values.astype(float)
    t2, dr = 1 - t1, (d.drug == "SalB").values.astype(float)
    D = [np.isclose(d.isi.values, g).astype(float) for g in grid]
    terms = {"tr x task 1": [x * t1 for x in D[1:]], "tr x task 2": [x * t2 for x in D[1:]],
             "tr x drug x task 1": [x * t1 * dr for x in D], "tr x drug x task 2": [x * t2 * dr for x in D]}
    return vta.partial_f(d.post.values, [np.ones(len(d)), t2], terms).astype({"df1": int, "df2": int})

a = tr[tr.type.between(1, 9)]
print("anova-time-task-drug-interaction without the free water term (paper: p 0.027, 0.0011, 0.62, 0.0062)")
print("(a) one mean per neuron and ISI, %d neurons" % a.neuron.nunique())
print(time_task_drug(a.groupby(["neuron", "task", "drug", "isi"], as_index=False).post.mean()).to_string(float_format="{:.3g}".format))
print("(b) single rewarded odor A trials, %d trials" % len(a))
print(time_task_drug(a).to_string(float_format="{:.3g}".format))

S0 = SL[("file", "trial")]
t1, dr = (S0.task == 1).values.astype(float), (S0.drug == "SalB").values.astype(float)
print("\nanova-slopes-task-drug on slopes not normalized by free water, %d neurons (paper: task p = 1.7e-5, task 2 x drug 0.032, task 1 x drug 0.94)" % len(S0))
print(vta.partial_f(S0.slope.values, [np.ones(len(S0))],
                    {"task": [1 - t1], "task 1 x drug": [t1 * dr], "task 2 x drug": [(1 - t1) * dr]}).astype({"df1": int, "df2": int}).to_string(float_format="{:.3g}".format))


Without the free water term, neither ANOVA is the paper's test, and these numbers decide no outcome.
On neuron means (reading a) no tr term is significant (p 0.41, 0.94, 0.99 and 0.87); on single trials
(reading b) all four are (p 1.3e-10, 0.021, 0.032 and 0.0086), including tr x drug x task 1, which the
paper finds not significant (p = 0.62). On unnormalized slopes, task is significant (p = 7.9e-06) and
task 1 x drug is not (p = 0.96), as in the paper, while task 2 x drug gives p = 0.066 against the paper's
0.032. Both claims stay blocked, `data_missing`.



## Summary

| Claim | Outcome |
|---|---|
| fig4a-task1-saline-post-decrease | reproduced |
| fig4b-task2-saline-post-increase | reproduced |
| fig4b-odorc-below-latest-odora | reproduced |
| fig4ab-saline-pre-decrease | reproduced |
| fig4c-task1-salb-post-decrease | reproduced |
| fig4d-task2-salb-flat | reproduced |
| fig4cd-salb-pre-decrease | reproduced |
| fig5f-task2-slope-signs | reproduced |
| fig5e-task1-significant-slopes-unchanged | reproduced (assumed: trial-by-trial baseline) |
| fig5f-task2-significant-slopes-shift | reproduced (assumed: trial-by-trial baseline) |
| fig4e-task2-slope-distribution-shift | reproduced |
| fig5e-task1-slope-distribution-unchanged | reproduced |
| fig5f-task2-slope-variance-up | reproduced |
| fig5e-task1-slope-variance-unchanged | reproduced |
| fig6ab-odorc-above-odorb | reproduced |
| fig6cd-omission-dips-intact | reproduced |
| figs5d-omission-unchanged-by-salb | reproduced |
| fig7cd-model-intact | reproduced (model ported to Python) |
| fig7fgh-model-belief-impaired | reproduced (model ported to Python) |
| fig7ijk-model-timing-impaired | reproduced (model ported to Python) |
| fig2-kord-suppresses-mpfc, figs1e-kord-abolishes-late-activity, figs6ce-kord-task1-animal | blocked, data_missing (mPFC units) |
| fig3g-anticipatory-licking, figs3-task1-licks-more-sooner, figs3-salb-spares-licking | blocked, data_missing (licks) |
| figs4-cell-types-unchanged | blocked, data_missing (unidentified VTA neurons) |
| figs5b-task2-salb-below-free-water, figs5c-odorbc-normalized, anova-slopes-task-drug, anova-time-task-drug-interaction | blocked, data_missing (free water deliveries) |
| figs6-salb-without-kord | blocked, data_missing (no-KORD control neurons) |

How the analysis differs from the paper's. The files hold no recording onset, so the 40-minute
window is read two ways (every trial in the file, or odor onsets within 40 min of the session's first
spike); every outcome is the same under both. The paper does not say whether baseline is subtracted
trial by trial; that reading recovers the paper's slope counts exactly and is adopted, and the two
claims on significant slopes rest on it. The model is a Python port of the paper's MATLAB code, with
numpy's random numbers in place of MATLAB's, so single simulations differ from the authors'.

Two blocked ANOVAs (anova-time-task-drug-interaction, anova-slopes-task-drug) need the free water
response. Their other terms are run without it, as a deviation that decides no outcome: the interaction
terms do not show the paper's pattern on either reading, and on unnormalized slopes task is significant
and task 1 x drug is not, as in the paper, while task 2 x drug gives p = 0.066 (paper 0.032). The timing
impaired model (fig7ijk) passes its criterion, but its omission dips are reduced to 25 to 43% of the
intact ones, still below zero, so "abolished" holds only in the criterion's weaker sense; this is asked of
a reader.

What was found about the dataset. The data reproduce the paper's statistics closely: most F values agree to two digits, and
the p values to their order of magnitude, about half of them to two digits. The deposit holds only the light-identified dopamine neurons: the licks, free
water deliveries, mPFC recordings, unidentified VTA neurons and the no-KORD control neurons that
twelve of the paper's claims use are not in it. No file names its mouse or session; sessions can be
recovered from shared odor onsets, mice cannot, so the per-mouse inset of Figure 5 cannot be made.
Three pairs of task 1 SalB files share most of their spikes and are very likely one neuron each,
deposited twice; the paper's count of 42 includes them, and counting each once changes no outcome.
21 file trials have a reward but no trial type; keeping them changes no result. The printed counts of Figure 5F (SalB 9 significant
negative slopes) give Fisher p = 0.037, while the data give 8 and p = 0.0405, the paper's 0.041.


In [ ]:
print(f"total runtime {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")